In [1]:
import os

os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [2]:


import ray

from autotsc import utils

2025-11-26 20:03:45.096564: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
X_train, y_train, X_test, y_test = utils.load_dataset("Meat")

In [4]:
from aeon.classification.deep_learning import IndividualLITEClassifier
# m = IndividualLITEClassifier(verbose=0)
# m.fit(X_train, y_train)
# pred = m.predict(X_test)
# accuracy_score(y_test, pred)

In [5]:
!uv pip install nvidia-ml-py3

import os
import time

import pynvml

Audited 1 package in 2ms


In [6]:
# Initialize NVML
pynvml.nvmlInit()


def gpu_mem_mb(gpu_id=0):
    handle = pynvml.nvmlDeviceGetHandleByIndex(gpu_id)
    info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    return info.used / (1024 * 1024)


def measure_gpu_usage_for_model():
    import tensorflow as tf

    # Measure BEFORE
    before = gpu_mem_mb()
    print("GPU before:", before, "MB")

    # Create and run the classifier with a few epochs
    m = IndividualLITEClassifier(verbose=0, n_epochs=30)
    m.fit(X_train, y_train)

    # Give TF time to flush GPU kernels
    time.sleep(0.5)

    after = gpu_mem_mb()
    print("GPU after:", after, "MB")
    del m
    tf.keras.backend.clear_session()
    return after - before


@ray.remote(num_gpus=1)  # reserve a GPU for this task
def measure_gpu_usage_for_model_ray():
    import time

    import pynvml
    import tensorflow as tf

    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)

    def gpu_mem():
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        return info.used / 1024 / 1024

    before = gpu_mem()
    print("GPU before:", before, "MB")

    # Run the model for a few epochs
    m = IndividualLITEClassifier(verbose=0, n_epochs=5)
    m.fit(X_train, y_train)
    time.sleep(0.5)

    after = gpu_mem()
    print("GPU after:", after, "MB")

    del m
    tf.keras.backend.clear_session()

    return after - before


print("GPU currently used:", gpu_mem_mb(), "MB")

ray.shutdown()
ray.init(num_gpus=1)

# mem_used = measure_gpu_usage_for_model()
# print("Estimated GPU memory per model:", mem_used, "MB")
# after_memory_usage = gpu_mem_mb()
# print(after_memory_usage)

gpu_mem_future = measure_gpu_usage_for_model_ray.remote()
mem_used = ray.get(gpu_mem_future)
print("Estimated GPU memory per model:", mem_used, "MB")

# Check GPU memory after the task finishes
# wait 10 seconds
pynvml.nvmlInit()

for _ in range(50):
    time.sleep(0.1)
    print("GPU currently used:", gpu_mem_mb(), "MB")
ray.shutdown()

GPU currently used: 356.125 MB


2025-11-26 20:03:50,898	INFO worker.py:2023 -- Started a local Ray instance.
/home/gasper_p/workspace/repos/AutoTSC/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
(pid=2267042) 2025-11-26 20:03:54.133110: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
(pid=2267042) To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


(measure_gpu_usage_for_model_ray pid=2267042) GPU before: 356.125 MB


(measure_gpu_usage_for_model_ray pid=2267042) 2025-11-26 20:03:55.710277: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
(measure_gpu_usage_for_model_ray pid=2267042) WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
(measure_gpu_usage_for_model_ray pid=2267042) I0000 00:00:1764183835.711182 2267042 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6164 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:04:00.0, compute capability: 8.6
(measure_gpu_usage_for_model_ray pid=2267042) 2025-11-26 20:03:57.429673: I external/local_xla/xla/service/service.cc:163] XLA service 0x78798c0079c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
(measure_gpu_usage_for_model_ray pid=2267042) 2025-11-26 20:03:57.429685: I externa

(measure_gpu_usage_for_model_ray pid=2267042) GPU after: 2307.25 MB
Estimated GPU memory per model: 1951.125 MB
GPU currently used: 2307.25 MB
GPU currently used: 2307.25 MB
GPU currently used: 2307.25 MB
GPU currently used: 2307.25 MB
GPU currently used: 2307.25 MB
GPU currently used: 2307.25 MB
GPU currently used: 2307.25 MB
GPU currently used: 2307.25 MB
GPU currently used: 2307.25 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 356.125 MB
GPU currently used: 

In [7]:
import pynvml as nv


def get_gpu_memory():
    nv.nvmlInit()
    handle = nv.nvmlDeviceGetHandleByIndex(0)
    info = nv.nvmlDeviceGetMemoryInfo(handle)

    return {
        "total_mb": info.total / 1024 / 1024,
        "used_mb": info.used / 1024 / 1024,
        "free_mb": info.free / 1024 / 1024,
    }


get_gpu_memory()

{'total_mb': 8192.0, 'used_mb': 356.125, 'free_mb': 7835.875}

In [8]:
dfdf = Dfdgdgf

NameError: name 'Dfdgdgf' is not defined

In [ ]:
from joblib import Parallel, delayed


def train_model(seed):
    m = IndividualLITEClassifier(verbose=1, random_state=seed)
    m.fit(X_train, y_train)
    return m.predict(X_test)


n_jobs = 4
# This will cause TensorFlow errors!
results = Parallel(n_jobs=n_jobs, backend="loky")(delayed(train_model)(i) for i in range(n_jobs))